# Fair Compensation Project

### Merging Datasets

Once, the data has been cleaned, we can merge it all into one dataset to be used in our clustering and regression steps. This step produces two different datasets. One that includes the data from rows in jobs.csv with no salary, and one that only includes data from rows in jobs.csv with salary data. The first dataset provides more data for clustering as rows without salary can still provide useful non-salary data. The second dataset is more suited for regression as price is needed for more accurate predictions.

In [1]:
import pandas as pd
import numpy as np

jobs = pd.read_csv('data/cleaned/jobs.csv')
companies = pd.read_csv('data/cleaned/companies.csv')
employment = pd.read_csv('data/cleaned/employment.csv')
pce = pd.read_csv('data/cleaned/pce.csv')

print("jobs:", jobs.shape)
print("companies:", companies.shape)
print("employment:", employment.shape)
print("pce:", pce.shape)

# join jobs and companies
df = jobs.merge (
    companies, 
    on = 'company_clean',
    how = 'left',
    suffixes = ('', '_co')
)

matched = df['sector'].notna().sum()
print(f"Jobs matched to a company: {matched} / {len(df)} ({round(matched/len(df)*100, 1)}%)")
print(f"Shape after jobs-company join: {df.shape}")

# join jobs and employment
df = df.merge (
    employment.rename(columns = {
        'occ_code': 'bls_occ_code',
        'annual_mean_wage': 'bls_annual_mean',
        'annual_median_wage': 'bls_annual_median',
        'annual_p10_wage': 'bls_p10',
        'annual_p90_wage': 'bls_p90',
        'total_unemployed': 'bls_total_unemployed',
    }),
    left_on = ['state', 'bls_occ_code'],
    right_on = ['state_abbr', 'bls_occ_code'],
    how = 'left'
)

matched = df['bls_annual_mean'].notna().sum()
print(f"Jobs matched to BLS wage: {matched} / {len(df)} ({round(matched/len(df)*100, 1)}%)")
print(f"Shape after jobs-BLS join: {df.shape}")

# prep state names
state_abbr_to_name = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'DC': 'District of Columbia', 'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming'
}

df['state_name'] = df['state_abbr'].map(state_abbr_to_name)
print("Unmapped states:", df[df['state_name'].isna()]['state_abbr'].unique())

# join jobs and pce
df = df.merge (
    pce.rename(columns = {
        'state': 'state_name',
        'pce_2023_in_millions': 'pce_2023_in_millions'
    }),
    on = 'state_name',
    how = 'left'
)

matched = df['pce_2023_in_millions'].notna().sum()
print(f"Jobs matched to PCE: {matched} / {len(df)} ({round(matched/len(df)*100, 1)}%)")
print(f"Shape after jobs-PCE join: {df.shape}")

print()
print("Merge Quality Summary")
print(f"Total rows: {len(df)}")
print(f"Rows with salary data:       {df['annual_mean'].notna().sum()}")
print(f"Rows with BLS match:         {df['bls_annual_mean'].notna().sum()}")
print(f"Rows with company match:     {df['sector'].notna().sum()}")
print(f"Rows with BEA match:         {df['pce_2023_in_millions'].notna().sum()}")
print()


# rows usable for fairness index (need all three)
usable = df[
    df['annual_mean'].notna() &
    df['bls_annual_mean'].notna() &
    df['pce_2023_in_millions'].notna()
]
print(f"Rows usable for fairness index (salary + employment + PCE): {len(usable)}")
print(f"As % of total: {round(len(usable)/len(df)*100, 1)}%")


df.to_csv('data/cleaned/merged.csv', index = False) # can use for clustering 
print(f"Saved merged.csv {df.shape}")

usable.to_csv('data/cleaned/modelling.csv', index = False) # can use for regression and fairness index calculation
print(f"Saved modelling.csv {usable.shape}")


jobs: (97352, 15)
companies: (32358, 10)
employment: (35470, 9)
pce: (50, 2)
Jobs matched to a company: 76933 / 97352 (79.0%)
Shape after jobs-company join: (97352, 24)
Jobs matched to BLS wage: 90923 / 97352 (93.4%)
Shape after jobs-BLS join: (97352, 32)
Unmapped states: <StringArray>
[nan]
Length: 1, dtype: str
Jobs matched to PCE: 86351 / 97352 (88.7%)
Shape after jobs-PCE join: (97352, 34)

Merge Quality Summary
Total rows: 97352
Rows with salary data:       40643
Rows with BLS match:         90923
Rows with company match:     76933
Rows with BEA match:         86351

Rows usable for fairness index (salary + employment + PCE): 36450
As % of total: 37.4%
Saved merged.csv (97352, 34)
Saved modelling.csv (36450, 34)


### Normalization of Salary

After our data has been merged into a single dataset, we can begin to normalize our salary values (since salary values are often skewed). In this step, is also is important to save our scaler for later so that we can "unscale" our salaries when calculating the fair compensation index. 

In [2]:
# import matplotlib.pyplot as plt

# df = pd.read_csv('data/cleaned/modelling.csv')
# print(df.shape)
# salary_cols = ['annual_mean', 'annual_low', 'annual_high', 'bls_annual_mean', 'bls_annual_median', 'bls_p10', 'bls_p90', 'pce_2023_in_millions',]

# for col in salary_cols:
#     if col in df.columns:
#         df[col]= pd.to_numeric(df[col], errors = 'coerce')

# print("\nSkewness before transformation:")
# for col in salary_cols:
#     if col in df.columns:
#         print(f" {col}: {df[col].skew():.3f}")
        
# fig, axes = plt.subplots(2, 4, figsize=(16, 8))
# axes = axes.flatten()

# for i, col in enumerate(salary_cols):
#     if col in df.columns:
#         axes[i].hist(df[col].dropna(), bins=50)
#         axes[i].set_title(col)
#         axes[i].set_xlabel('Value')

# plt.suptitle('Salary distributions before normalization')
# plt.tight_layout()
# plt.savefig('distributions_before.png')
# plt.show()


In [3]:
# print(df[['bls_annual_mean', 'bls_annual_median', 'bls_p10']].dtypes)
# print(df[['bls_annual_mean', 'bls_annual_median', 'bls_p10']].head(10))

# for col in salary_cols:
#     if col in df.columns:
#         df[col]= pd.to_numeric(df[col], errors = 'coerce')
        
# for col in salary_cols:
#     if col in df.columns:
#         log_col = f'log_{col}'
#         df[log_col] = np.log1p(df[col])

# log_cols = [f'log_{col}' for col in salary_cols if col in df.columns]
# print("\nSkewness before transformation:")
# for col in log_cols:
#         print(f" {col}: {df[col].skew():.3f}")

In [4]:
# from sklearn.preprocessing import StandardScaler
# import joblib
# import os

# os.makedirs('outputs', exist_ok=True)

# scalers = {}
# scaled_cols = []

# for log_col in log_cols:
#     col_name = log_col.replace('log_', '')
#     scaled_col = f'scaled_{col_name}'
#     scaled_cols.append(scaled_col)
    
#     # fit scaler only on non-null values
#     non_null_mask = df[log_col].notna()
#     print(f"{log_col}: {non_null_mask.sum()} non-null rows")
    
#     scaler = StandardScaler()
#     scaler.fit(df.loc[non_null_mask, [log_col]])
    
#     # keep NaN column values as NaN
#     df[scaled_col] = np.nan
#     df.loc[non_null_mask, scaled_col] = scaler.transform(
#         df.loc[non_null_mask, [log_col]]
#     ).flatten()
    
#     scalers[col_name] = scaler

# print("\nScaled column stats (should be ~mean=0, std=1):")
# print(df[scaled_cols].describe().round(3))

In [5]:
# import joblib
# import os

# os.makedirs('outputs', exist_ok=True)

# # save scaler for inverse_transform predictions (for the fairness index calculation!) later
# joblib.dump(scalers, 'outputs/salary_scalers.pkl')
# print("Scaler saved to outputs/salary_scalers.pkl")
# print("\nScalers fitted on:")
# for col, scaler in scalers.items():
#     print(f"  {col}: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}")


In [6]:
# fig, axes = plt.subplots(2, 4, figsize=(16, 8))
# axes = axes.flatten()

# for i, col in enumerate(scaled_cols):
#     if col in df.columns:
#         axes[i].hist(df[col].dropna(), bins=50)
#         axes[i].set_title(col)
#         axes[i].set_xlabel('Value')

# plt.suptitle('Salary distributions after log and z-score normalization')
# plt.tight_layout()
# plt.savefig('distributions_after.png')
# plt.show()

# df.to_csv('data/cleaned/modelling_normalized.csv', index=False)
# print(f"Saved modelling_normalized.csv: {df.shape}")
# print(f"\nOriginal salary columns retained: {salary_cols}")
# print(f"Log transformed columns added: {log_cols}")
# print(f"Scaled columns added: {scaled_cols}")